In [ ]:
# # VISULAIZER V1 TESTER DOESNT WORK ANYMORE PROBABLY
# import sys, os
# project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
# sys.path.append(project_root)
# import numpy as np
# import matplotlib.pyplot as plt
# from IPython.display import HTML
# from bicycle_model_visualizer import VehicleBicycleVisualizer, create_animation
# from bicycle_model_dynamics import VehicleBicycleModel
# from bicycle_model_sim import simulate_bicycle_model
# from planners.env import OtherVehicle, lane_center_y

# # -------------------------------
# # 1. Define Ego Vehicle
# # -------------------------------
# m = 1000
# Iz = 2500
# Lf = 1.1
# Lr = 1.1
# Cf = 80000
# Cr = 80000
# d_limit = np.deg2rad(30)

# bicycle = VehicleBicycleModel(m, Iz, Lf, Lr, Cf, Cr, d_limit)

# x0 = np.array([0.0, 0.0, 0.0, 5.0, 0.0, 0.0])  # X, Y, psi, vx, vy, w
# tf = 5.0  # seconds

# x_traj, u_traj, t_traj = simulate_bicycle_model(x0, tf, bicycle)

# # -------------------------------
# # 2. Define Traffic Vehicles (using env.OtherVehicle)
# # -------------------------------

# # Two vehicles: one in lane 2 (upper lane), one in lane 0 (lower lane)
# traffic = [
#     OtherVehicle(x=5.0,  y=lane_center_y(2), v=2.0, lane=2),
#     OtherVehicle(x=-5.0, y=lane_center_y(0), v=3.0, lane=0)
# ]

# # -------------------------------
# # 3. Initialize Visualizer
# # -------------------------------
# vis = VehicleBicycleVisualizer(show_velocity=True, show_trail=True)
# vis.add_traffic(traffic, length=4.0, width=1.8)

# # -------------------------------
# # 4. Prepare trajectories for visualizer
# # -------------------------------
# x_vis = x_traj.T  # shape (N, 6)
# u_vis = u_traj.T  # shape (N, 2)
# dt = t_traj[1] - t_traj[0]

# # -------------------------------
# # 5. Create Animation
# # -------------------------------
# ani = create_animation(vis, x_vis, u_vis, dt)

# # -------------------------------
# # 6. Display in notebook
# # -------------------------------
# HTML(ani.to_jshtml())


In [ ]:
# VISUALIZER V2 TEST - SINUSOIDAL PATH DUMMY TRAJECTORIES
import sys
import os
sys.path.append(os.path.abspath('..'))
import numpy as np
from planners.env import RoadGeometry, World_curves, lane_xy
from bicycle_model_visualizer import VehicleBicycleVisualizer, create_animation
from IPython.display import HTML

dt = 0.05
T = 10
N = int(T / dt)

# road LONG CURVE CURVE
R = 100      
theta_max = np.pi/2
N = 200      
theta = np.linspace(0, theta_max, N)
xs = R * np.sin(theta)
ys = R * (1 - np.cos(theta))

# road SINUSOIDAL
# xs = np.linspace(0, 200, 200)
# ys = 20*np.sin(xs * 0.03)

road = RoadGeometry(xs, ys)

# world with curved traffic
world = World_curves(road, num_lanes=3, moving_lanes=[0,2])

# create visualizer & register traffic vehicles
vis = VehicleBicycleVisualizer(show_velocity=False, show_trail=True)
vis.add_traffic(world.traffic)

# ego trajectory (dummy example)
x_traj = np.zeros((6, N))
u_traj = np.zeros((2, N))
x_traj[0, :] = np.linspace(0, 150, N)      # X
x_traj[1, :] = lane_xy(road, x_traj[0,:], 1, 3)[1]  # center lane Y
x_traj[2, :] = 0.0                          # yaw

# traffic logs
M = len(world.traffic)
obstacle_trajs = np.zeros((N, M, 5))

for i in range(N):
    for j, car in enumerate(world.traffic):
        obstacle_trajs[i, j, 0] = car.x
        obstacle_trajs[i, j, 1] = car.y
        obstacle_trajs[i, j, 2] = car.v
        obstacle_trajs[i, j, 3] = car.lane
        obstacle_trajs[i, j, 4] = car.s
    world.step(dt)

# animate
# Determine frame limits
all_x = np.concatenate([x_traj[0,:], obstacle_trajs[:,:,0].flatten()])
all_y = np.concatenate([x_traj[1,:], obstacle_trajs[:,:,1].flatten()])
padding = 5.0

vis.ax.set_xlim(all_x.min() - padding, all_x.max() + padding)
vis.ax.set_ylim(all_y.min() - padding, all_y.max() + padding)
vis.ax.set_aspect("equal")

ani = create_animation(vis, x_traj, u_traj, obstacle_trajs, dt)
HTML(ani.to_jshtml())


In [ ]:
import os, sys
from pathlib import Path

# Find project root dynamically
cwd = Path(os.getcwd())

for p in [cwd, cwd.parent, cwd.parent.parent]:
    if (p / "planners").exists() and (p / "controllers").exists():
        project_root = p
        break
else:
    raise RuntimeError("Could not find project root containing planners/ and controllers/")

print("Project root:", project_root)

# Add to Python path
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "planners"))
sys.path.insert(0, str(project_root / "controllers"))
sys.path.insert(0, str(project_root / "Bicycle_Model"))


In [ ]:
!pip install drake
#!pip install pydrake

In [ ]:
import numpy as np
import sys
from dataclasses import dataclass
import matplotlib as mpl
from IPython.display import HTML

from bicycle_model_dynamics import VehicleBicycleModel
from bicycle_model_visualizer import VehicleBicycleVisualizer, create_animation
from behavior_planner import BehaviorPlanner
from reference_provider import ReferenceProvider

# Attempt MPC import; fallback to simple PD controller
MPC_AVAILABLE = True
try:
    from mpc_controller import DrakeMPC, DrakeMPCConfig
except Exception as e:
    MPC_AVAILABLE = False
    print("MPC unavailable; using fallback PD controller:", e)


In [ ]:
import math

def wrap_to_pi(a):
    return (a + math.pi) % (2*math.pi) - math.pi

class FallbackController:
    def __init__(self, steer_limit_rad, a_limit=3.0):
        self.k_y = 1.2
        self.k_psi = 2.0
        self.k_v = 1.5
        self.d_limit = steer_limit_rad
        self.a_limit = a_limit

    def compute(self, x6, ref6_horizon):
        xr, yr, psir, vref = ref6_horizon[0,0], ref6_horizon[1,0], ref6_horizon[2,0], ref6_horizon[3,0]
        X, Y, psi, vx, vy, w = x6
        e_y = yr - Y
        e_psi = wrap_to_pi(psir - psi)
        e_v = vref - vx
        delta = self.k_y * e_y + self.k_psi * e_psi
        a = self.k_v * e_v
        delta = float(np.clip(delta, -self.d_limit, self.d_limit))
        a = float(np.clip(a, -self.a_limit, self.a_limit))
        return np.array([delta, a], dtype=float)

class ControllerWrapper:
    def __init__(self, ref_provider, bicycle_model):
        self.ref_provider = ref_provider
        self.mpc_available = MPC_AVAILABLE
        if self.mpc_available:
            self.mpc = DrakeMPC(reference_provider=ref_provider)
            print("Using Drake MPC controller")
        else:
            self.fallback = FallbackController(steer_limit_rad=bicycle_model.d_limit)
            print("Using fallback PD controller")

    def compute_control(self, x6):
        if self.mpc_available:
            ref6 = self.ref_provider.get_ref_horizon(x0=x6, T=self.mpc.cfg.T, DT=self.mpc.cfg.DT)
            return self.mpc.solve_mpc(x6, ref6)
        else:
            ref6 = self.ref_provider.get_ref_horizon(x0=x6, T=8.0, DT=0.1)
            return self.fallback.compute(x6, ref6)

In [ ]:
# Closed-loop integration
from dataclasses import dataclass

def integrate_step(model, x, u, dt):
    return x + dt * model.continuous_time_full_dynamics(x, u)

@dataclass
class SimConfig:
    dt: float = 0.02
    tf: float = 8.0

MODEL_PARAMS = dict(
    m=1000.0,
    Iz=2500.0,
    Lf=1.2,
    Lr=1.2,
    Cf=80000.0,
    Cr=80000.0,
    d_limit=float(np.deg2rad(30.0)),
)

def integrate_step(model, x, u, dt):
    return x + dt * model.continuous_time_full_dynamics(x, u)

def run_closed_loop_sim(cfg=SimConfig()):
    bicycle = VehicleBicycleModel(**MODEL_PARAMS)
    bp = BehaviorPlanner(preferred_lane=1, cruise_speed=20.0, bicycle=bicycle)
    ref_provider = ReferenceProvider(behavior_planner=bp)
    ctrl = ControllerWrapper(ref_provider, bicycle)

    x = np.array([0.0, -3.7, 0.0, 15.0, 0.0, 0.0])
    steps = int(np.ceil(cfg.tf / cfg.dt))
    x_hist = np.zeros((steps + 1, 6))
    u_hist = np.zeros((steps + 1, 2))
    t_hist = np.zeros((steps + 1,))
    x_hist[0] = x
    t_hist[0] = 0.0

    mpc_period = 0.1
    next_mpc_time = 0.0
    u_cur = np.array([0.0, 0.0])
    t = 0.0
    for k in range(1, steps + 1):
        if t >= next_mpc_time:
            u_cur = ctrl.compute_control(x)
            next_mpc_time += mpc_period
        x = integrate_step(bicycle, x, u_cur, cfg.dt)
        t += cfg.dt
        x_hist[k] = x
        u_hist[k] = u_cur
        t_hist[k] = t
    return x_hist, u_hist, t_hist

In [ ]:
if __name__ == "__main__":
    x_hist, u_hist, t_hist = run_closed_loop_sim()
    print("States shape:", x_hist.shape)
    print("Inputs shape:", u_hist.shape)
    print("Time shape:", t_hist.shape)

    x_vis = x_hist.T
    u_vis = u_hist.T
    N = u_hist.shape[0]
    obstacle_trajs = np.zeros((N, 0, 4))  # empty obstacles

    vis = VehicleBicycleVisualizer(show_velocity=True, show_trail=True)
    ani = create_animation(vis, x_vis, u_vis, obstacle_trajs, dt=(t_hist[1]-t_hist[0]))
    mpl.rcParams['animation.html'] = 'jshtml'
    html = HTML(ani.to_jshtml())

In [ ]:
import numpy as np
from typing import Optional
from planners.env import lane_center_y
from behavior_planner import BehaviorPlanner
from planners.env import OtherVehicle
from planners.perception import NoisyPerception
from reference_provider import ReferenceProvider
from mpc_controller import DrakeMPC
from planners.obstacle_spawner import spawn_random_obstacles


class ObstacleAwarePlanner(BehaviorPlanner):
    """
    Behavior planner wrapper that supports multiple obstacles:
    - Triggers lane-change using distance and TTC thresholds
    - Slows to match obstacle speed when boxed in
    - Adds hysteresis + commitment to avoid bouncing between lanes
    """

    def __init__(self, preferred_lane=1, cruise_speed=20.0, bicycle=None):
        super().__init__(preferred_lane, cruise_speed, bicycle)
        self.obstacles = []  # list[OtherVehicle]

        # Safety / decision thresholds
        self.safety_distance = 30.0      # meters (earlier trigger)
        self.ttc_threshold = 2.0         # seconds (time-to-collision)
        self.slow_down_distance = 30.0   # follow distance for speed matching

        # Hysteresis / commitment
        self.default_cruise = cruise_speed
        self.lane_change_committed = False    # True once we decide to change
        self.current_target_lane = None       # Lane we are moving toward
        self.eval_candidate_lane = None       # Lane currently being evaluated
        self.eval_counter = 0                 # How many consecutive frames it stayed best
        self.required_stability_steps = 5     # frames of consistent best lane before commit
        self.post_change_cooldown = 1.0       # seconds after change before considering next
        self.cooldown_timer = 0.0

    def _y_to_lane(self, y):
        centers = [lane_center_y(0), lane_center_y(1), lane_center_y(2)]
        diffs = np.abs(np.array(centers) - y)
        return int(np.argmin(diffs))

    def _nearest_ahead_per_lane(self, ego_x, ego_vx):
        lane_info = {
            0: dict(dx=np.inf, v=None, ttc=np.inf),
            1: dict(dx=np.inf, v=None, ttc=np.inf),
            2: dict(dx=np.inf, v=None, ttc=np.inf),
        }
        for obs in (self.obstacles or []):
            lane = self._y_to_lane(obs.y)
            dx = obs.x - ego_x
            if dx <= 0:
                continue  # behind or at same x
            v_rel = ego_vx - obs.v
            ttc = dx / v_rel if v_rel > 0.0 else np.inf
            if dx < lane_info[lane]['dx']:
                lane_info[lane] = dict(dx=dx, v=obs.v, ttc=ttc)
        return lane_info

    def _score_lane(self, info, lane_id):
        # Larger dx and TTC are better; slight bias for center lane
        dx_score = (info['dx'] if np.isfinite(info['dx']) else 1e6) * 0.5
        ttc_score = (info['ttc'] if np.isfinite(info['ttc']) else 1e3) * 1.0
        center_bonus = 5.0 if lane_id == 1 else 0.0
        return dx_score + ttc_score + center_bonus

    def _select_best_adjacent_lane(self, current_lane, lane_info):
        candidates = [l for l in [current_lane - 1, current_lane + 1] if 0 <= l <= 2]
        if not candidates:
            return None
        best_lane = None
        best_score = -np.inf
        for l in candidates:
            score = self._score_lane(lane_info[l], l)
            if score > best_score:
                best_score = score
                best_lane = l
        return best_lane

    def update(self, ego_x: float, ego_y: float, ego_vx: float, dt: float):
        # Baseline: reset cruise speed each step and apply following logic
        self.cruise_speed = self.default_cruise

        # Cooldown after a completed lane change
        if self.cooldown_timer > 0.0:
            self.cooldown_timer = max(0.0, self.cooldown_timer - dt)

        current_lane = self._y_to_lane(ego_y)
        lane_info = self._nearest_ahead_per_lane(ego_x, ego_vx)
        cur = lane_info[current_lane]

        # Following behavior: start matching obstacle speed if too close
        if cur['dx'] < self.slow_down_distance and cur['v'] is not None and (ego_vx > cur['v']):
            self.cruise_speed = min(self.cruise_speed, max(cur['v'] - 0.5, 0.0))

        # If we are already committed to a lane change, do not re-evaluate.
        if self.lane_change_committed:
            # Let base BehaviorPlanner execute the maneuver toward preferred_lane.
            # Consider the lane change complete once the inferred lane matches target.
            if current_lane == self.current_target_lane:
                self.lane_change_committed = False
                self.current_target_lane = None
                self.eval_candidate_lane = None
                self.eval_counter = 0
                self.cooldown_timer = self.post_change_cooldown
            return super().update(ego_x, ego_y, ego_vx, dt)

        # Hazard in current lane?
        hazard = (cur['dx'] < self.safety_distance) or (cur['ttc'] < self.ttc_threshold)

        # Only consider new lane change if no cooldown
        if hazard and self.cooldown_timer <= 0.0:
            best_lane = self._select_best_adjacent_lane(current_lane, lane_info)

            if best_lane is not None and best_lane != current_lane:
                # Hysteresis: require the same best lane for several consecutive steps
                if best_lane != self.eval_candidate_lane:
                    # New candidate lane: reset counter
                    self.eval_candidate_lane = best_lane
                    self.eval_counter = 1
                else:
                    self.eval_counter += 1

                # Once stable enough, commit to lane change
                if self.eval_counter >= self.required_stability_steps:
                    self.preferred_lane = best_lane
                    self.current_target_lane = best_lane
                    self.lane_change_committed = True
            else:
                # No good adjacent lane or same lane; reset evaluation
                self.eval_candidate_lane = None
                self.eval_counter = 0
        else:
            # No hazard or still cooling down: reset evaluation
            self.eval_candidate_lane = None
            self.eval_counter = 0

        # Call base BehaviorPlanner to generate TrajectoryRequest toward preferred_lane
        return super().update(ego_x, ego_y, ego_vx, dt)


def run_overtake_sim_ttc(
    tf: float = 30.0,
    dt: float = 0.05,
    ego_speed: float = 15.0,
    obstacles_init=None,
    random_seed: Optional[int] = None,
    num_random_obstacles: int = 5,
    min_front_offset: float = 5.0,
    guaranteed_obstacle_lane: Optional[int] = 1,
    ):
    """
    Runs the overtaking scenario with multiple obstacles and TTC-based lane-change.
    Returns: x_traj, u_traj, t_traj, obstacle_trajs (N+1, M, 4), dt, N
    """
    # Vehicle model params
    m = 1000
    Iz = 2500
    Lf = 1.1
    Lr = 1.6
    Cf = 80000
    Cr = 80000
    d_limit = np.deg2rad(30)

    bicycle = VehicleBicycleModel(m, Iz, Lf, Lr, Cf, Cr, d_limit)

    # Initial state: ego behind obstacles
    x0 = np.array([0.0, lane_center_y(1), 0.0, ego_speed, 0.0, 0.0])

    # Obstacles across all lanes (default set if none provided)
    if obstacles_init is None:
        min_speed = 6.0
        max_speed = max(ego_speed - 2.0, min_speed + 1.0)
        obstacles_init = spawn_random_obstacles(
            num_obstacles=num_random_obstacles,
            lanes=(0, 1, 2),
            x_range=(15.0, 55.0),
            speed_range=(min_speed, max_speed),
            min_gap=10.0,
            seed=random_seed,
            ego_x=x0[0],
            min_front_offset=min_front_offset,
        )
        
    if guaranteed_obstacle_lane is not None:
        ahead_threshold = x0[0] + min_front_offset
        has_lane_leader = any(
            obs.x > ahead_threshold and obs.lane == guaranteed_obstacle_lane
            for obs in obstacles_init
        )
        if not has_lane_leader:
            lane_y = lane_center_y(guaranteed_obstacle_lane)
            nominal_speed = max(6.0, min(ego_speed - 3.0, ego_speed - 1.0))
            obstacles_init.append(
                OtherVehicle(
                    x=ahead_threshold + 10.0,
                    y=lane_y,
                    v=nominal_speed,
                    lane=guaranteed_obstacle_lane,
                )
            )
            
    obstacles = [OtherVehicle(x=o.x, y=o.y, v=o.v, lane=o.lane) for o in obstacles_init]

    perception = NoisyPerception(
        obstacles,
        perception_dt=0.2,
        sigma_x=0.5,
        sigma_y=0.3,
    )

    # Planner and MPC
    bp = ObstacleAwarePlanner(preferred_lane=1, cruise_speed=ego_speed, bicycle=bicycle)
    bp.obstacles = perception.perceived_obstacles
    ref_provider = ReferenceProvider(bp, horizon_s=5.0)
    mpc = DrakeMPC(reference_provider=ref_provider)

    N = int(tf / dt)
    t_traj = np.linspace(0.0, tf, N + 1)
    x_traj = np.zeros((N + 1, 6))
    u_traj = np.zeros((N, 2))
    M = len(obstacles)
    obstacle_trajs = np.zeros((N + 1, M, 4))  # x, y, v, lane per obstacle

    x_traj[0] = x0
    for i, o in enumerate(obstacles):
        obstacle_trajs[0, i, :] = [o.x, o.y, o.v, o.lane]

    for k in range(N):
        x = x_traj[k]

        # Control via MPC (fallback: gentle centering accel)
        try:
            ref = ref_provider.get_ref_horizon(x0=x, T=mpc.cfg.T, DT=mpc.cfg.DT)
            u = mpc.solve_mpc(x, ref)
        except Exception as e:
            print("MPC FAILED:", e)
            target_lane_y = lane_center_y(bp.preferred_lane)
            y_error = target_lane_y - x[1]
            u = np.array([0.0, np.clip(0.3 * y_error, -0.3, 0.3)])

        u_traj[k] = u

        # Euler integrate
        xdot = bicycle.continuous_time_full_dynamics(x, u)
        x_next = x + dt * xdot
        x_traj[k + 1] = x_next

        # Advance obstacles (true world)
        for i, o in enumerate(obstacles):
            o.step(dt)
            obstacle_trajs[k + 1, i, :] = [o.x, o.y, o.v, o.lane]

        # Update noisy, low-rate perception after world moves
        perception.step(dt)
        bp.obstacles = perception.perceived_obstacles

    return x_traj, u_traj, t_traj, obstacle_trajs, dt, N


if __name__ == "__main__":
    x_traj, u_traj, t_traj, obstacle_trajs, dt, N = run_overtake_sim_ttc()
    print("Sim complete.")
    print("x_traj:", x_traj.shape, "u_traj:", u_traj.shape, "t:", t_traj.shape, "obstacles:", obstacle_trajs.shape)

In [ ]:
import numpy as np
from Bicycle_Model.bicycle_model_visualizer import VehicleBicycleVisualizer
from planners.env import lane_center_y, LANE_WIDTH
from matplotlib.animation import FuncAnimation
import matplotlib as mpl
from IPython.display import HTML

class SimpleVehicle:
    def __init__(self, x: float, y: float, psi: float = 0.0):
        self.x = x
        self.y = y
        self.psi = psi 
        
def create_overtake_animation(x_traj, u_traj, t_traj, obstacle_trajs, dt):
    """Create animation for overtaking scenario with multiple obstacles.
    Parameters
    ----------
    x_traj : ndarray (N+1, 6) or (6, N+1)
        Ego states over time. Will transpose if needed.
    u_traj : ndarray (N, 2) or (2, N)
        Control inputs over time. Will transpose if needed.
    t_traj : ndarray (N+1,)
        Time stamps.
    obstacle_trajs : ndarray (N+1, M, 4)
        Per-obstacle logs (x,y,v,lane).
    dt : float
        Time step.
    Returns
    -------
    fig, ani : matplotlib Figure and FuncAnimation
    """
    # Normalize shapes
    if x_traj.shape[0] != 6 and x_traj.shape[1] == 6:
        x_traj = x_traj.T
    if u_traj.shape[0] != 2 and u_traj.shape[1] == 2:
        u_traj = u_traj.T

    N = u_traj.shape[1]
    # Match ego frames length
    x_vis = x_traj[:, :N]  # (6, N)
    u_vis = u_traj[:, :N]  # (2, N)

    if obstacle_trajs.ndim != 3 or obstacle_trajs.shape[0] < N:
        raise ValueError("obstacle_trajs must be (N+1, M, 4) with N+1 >= u_traj frames")

    M = obstacle_trajs.shape[1]

    vis = VehicleBicycleVisualizer(show_velocity=True, show_trail=True)

    # Create traffic vehicle placeholders with psi attribute
    traffic_vehicles = [SimpleVehicle(obstacle_trajs[0, i, 0], obstacle_trajs[0, i, 1], 0.0) for i in range(M)]
    vis.add_traffic(traffic_vehicles)

    fig = vis.fig
    ax = vis.ax

    # Draw lane centerlines
    for lane_id in [0, 1, 2]:
        yc = lane_center_y(lane_id)
        ax.axhline(y=yc, color='yellow', linestyle='--', linewidth=1.0, alpha=0.6)

    # Limits
    obs_x_all = obstacle_trajs[:, :, 0]
    x_min = min(x_vis[0, :].min(), obs_x_all.min()) - 20
    x_max = max(x_vis[0, :].max(), obs_x_all.max()) + 20
    y_min = lane_center_y(0) - LANE_WIDTH
    y_max = lane_center_y(2) + LANE_WIDTH
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_title('Autonomous Lane Change - Multi-Obstacle')

    def update(frame):
        # Update traffic states for this frame (no heading dynamics assumed)
        for i in range(M):
            traffic_vehicles[i].x = obstacle_trajs[frame, i, 0]
            traffic_vehicles[i].y = obstacle_trajs[frame, i, 1]
            traffic_vehicles[i].psi = 0.0

        vis.draw(x_vis[:, frame], u_vis[:, frame], frame * dt)

        # Scroll window following ego
        x_ego = x_vis[0, frame]
        window_width = 60
        ax.set_xlim(x_ego - 20, x_ego + window_width)
        return []

    ani = FuncAnimation(fig, update, frames=N, interval=dt * 1000, blit=False, repeat=True)
    return fig, ani


if __name__ == "__main__":
    x_traj, u_traj, t_traj, obstacle_trajs, dt, N = run_overtake_sim_ttc()
    fig, ani = create_overtake_animation(x_traj, u_traj, t_traj, obstacle_trajs, dt)
    from IPython.display import HTML
    mpl.rcParams['animation.html'] = 'jshtml'
    html = HTML(ani.to_jshtml())
    display(html)


Intersection Maneuver

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath("..")  # or the path to MEAM517_Final_Project
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)


In [ ]:
# main_intersection_demo_notebook.py
import matplotlib as mpl
from IPython.display import HTML, display

from controllers.intersection_sim import run_intersection_sim
from controllers.intersection_animation import create_intersection_animation

def main():
    # Run the intersection simulation
    x_traj, u_traj, t_traj, dt, N, obs = run_intersection_sim(
        tf=12.0,
        dt=0.05,
        direction="east"
    )

    # Create animation
    fig, ani = create_intersection_animation(x_traj, u_traj, dt,obstacle_trajs=obs)

    # Make animation notebook-friendly
    mpl.rcParams['animation.html'] = 'jshtml'
    return HTML(ani.to_jshtml())

# Render directly in notebook
html = main()
display(html)
